In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import warnings
import os
warnings.filterwarnings('ignore')

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam



In [ ]:
# ! pip install -q kagglehub


In [ ]:
import kagglehub

path = kagglehub.dataset_download("samuelcortinhas/cats-and-dogs-image-classification")

In [ ]:
train_dir = f"{path}/train"
val_dir = f"{path}/test"

print(f"Train dir: {train_dir}")
print(f"Val dir: {val_dir}")
print(f"Train folders: {os.listdir(train_dir)}")
print(f"Val folders: {os.listdir(val_dir)}")

In [ ]:

# ══════════════════════════════════════════════════════════
#  DATA GENERATORS
# ══════════════════════════════════════════════════════════
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_iterator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    shuffle=True
)

val_iterator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

print(f"\nClass mapping: {train_iterator.class_indices}")

In [ ]:
# ══════════════════════════════════════════════════════════
#  VERIFY - Show sample images
# ══════════════════════════════════════════════════════════
plt.figure(figsize=(12, 4))
batch_x, batch_y = next(train_iterator)

for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(batch_x[i])
    label = 'Dog' if batch_y[i] == 1 else 'Cat'
    plt.title(label)
    plt.axis('off')

plt.tight_layout()
plt.show()

train_iterator.reset()

In [ ]:


# ══════════════════════════════════════════════════════════
#  MODEL
# ══════════════════════════════════════════════════════════
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128, 128, 3)
)

base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:

# ══════════════════════════════════════════════════════════
#  TRAINING
# ══════════════════════════════════════════════════════════
early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, mode='max')
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)

history = model.fit(
    train_iterator,
    epochs=20,
    validation_data=val_iterator,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
# ══════════════════════════════════════════════════════════
# RESULTS
# ══════════════════════════════════════════════════════════
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(acc, 'b-o', label='Train')
plt.plot(val_acc, 'r-o', label='Val')
plt.title(f'Accuracy (Best: {max(val_acc)*100:.1f}%)')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(loss, 'b-o', label='Train')
plt.plot(val_loss, 'r-o', label='Val')
plt.title('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"\n Best Validation Accuracy: {max(val_acc)*100:.2f}%")


In [ ]:
# ══════════════════════════════════════════════════════════
#  TEST
# ══════════════════════════════════════════════════════════
from google.colab import files

print("\n Upload test image:")
uploaded = files.upload()

for filename in uploaded.keys():
    img = load_img(filename, target_size=(128, 128))
    img_array = np.array(img) / 255.0
    img_array = img_array.reshape(1, 128, 128, 3)
    
    pred = model.predict(img_array, verbose=0)[0][0]
    
    label = f' Dog ({pred*100:.1f}%)' if pred > 0.5 else f' Cat ({(1-pred)*100:.1f}%)'
    
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(label, fontsize=16)
    plt.axis('off')
    plt.show()